# Surface-water mapping from Sentinel-1: dataset, TerraMind training, inference

Pipeline of this repository, end to end:

1. **Dataset** - Sentinel-1 RTC (Planetary Computer) -> refined Lee -> edge + binary Otsu, labelled with the **JRC Monthly
   History map of the same calendar month** (Google Earth Engine). Pixels where JRC and Otsu disagree are ignored.
2. **Training** - TerraMind (`S1RTC` modality) + light decoder on those labels. Whole sites can be held out.
3. **Inference** - any date: Otsu + TerraMind probability, fused; area, GeoTIFF export, time series.

Two modes (set `MODE` in the first code cell):

| `MODE` | Data | Needs | Purpose |
|---|---|---|---|
| `"demo"` | synthetic speckled scenes + a synthetic JRC | nothing (CPU, a few minutes) | check that the whole notebook works; random-init tiny model, so the numbers mean nothing about real accuracy |
| `"real"` | Planetary Computer + Earth Engine | network, `earthengine authenticate`, ideally a GPU | the real thing |

Labelled dates are **2017-2021** (Sentinel-1 RTC starts in 2017, JRC GSW1_4 ends in Dec 2021); the trained model then runs on any date.

## 0. Setup

In [ ]:
import os, sys, json, subprocess, tempfile
from pathlib import Path

# Colab: install the dependencies. numpy is pinned below 2.3 because Colab's preinstalled numba requires it; without the pin pip
# upgrades numpy and prints "numba requires numpy<2.3" (and numba then fails to import). If numpy was already imported in this
# session, restart the runtime once after this cell (Runtime > Restart session) and run the notebook again.
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy<2.3", "scipy", "scikit-image", "shapely", "pandas",
                    "matplotlib", "rasterio", "affine", "pyproj", "pystac-client", "planetary-computer", "odc-stac",
                    "earthengine-api", "torch", "terratorch"], check=True)

# repository root: hard-coded (edit this one line if the project lives elsewhere)
ROOT = Path("/content/drive/MyDrive/Surface_Water/project")
if "google.colab" in sys.modules and not ROOT.exists():
    from google.colab import drive
    drive.mount("/content/drive")
assert (ROOT / "angat_water").is_dir(), f"angat_water/ not found in {ROOT}: check the path above and that Drive is mounted"
sys.path.insert(0, str(ROOT))
print("repo:", ROOT)

## 1. Configuration

In [ ]:
MODE = "demo"                       # "demo" (offline, synthetic) | "real" (Planetary Computer + Earth Engine)

# ---- sites and labels
SITES = ["angat", "magat"]          # registered in angat_water/sites.py; add rivers with sites.custom(name, bbox, kind="river")
HOLDOUT = ["magat"]                 # whole sites kept out of training and used for validation ([] = east-strip block split)
START, END = "2018-01-01", "2021-12-31"     # Sentinel-1 dates to label (JRC GSW1_4 ends in Dec 2021)
GEE_PROJECT = None                  # your Google Cloud project for Earth Engine
MAX_MONTH_OFFSET = 0                # 0 = the JRC month must be the Sentinel-1 month (recommended)
MIN_AGREEMENT = 0.9                 # JRC-Otsu agreement a scene needs to be used for training

# ---- model
BACKBONE = "terramind_v1_base"      # tiny | small | base | large
PRETRAINED = True                   # downloads ibm-esa-geospatial/TerraMind-1.0-* from Hugging Face
UNFREEZE_LAST = 0                   # 0 = train the decoder only; N > 0 also fine-tunes the last N transformer blocks
PATCH, EPOCHS, BATCH, LR = 224, 15, 8, 3e-4
KEEP_EMPTY = 0.2                    # share of near-waterless patches kept (water is rare; rivers especially)

# ---- inference
INFER_SITE = "magat"
INFER_FROM, INFER_TO = "2024-01-01", "2024-03-31"   # any dates; they need no JRC
RUN_TIMESERIES = True
RES_M = 10.0

OUT = Path(tempfile.mkdtemp(prefix="water_")) if MODE == "demo" else ROOT / "outputs"
if MODE == "demo":                  # tiny, offline settings so the notebook finishes on a laptop CPU
    BACKBONE, PRETRAINED, PATCH, EPOCHS, BATCH, LR = "terramind_v1_tiny", False, 128, 4, 8, 1e-3
    START, END = "2019-03-01", "2019-09-30"
DATA, MODEL_DIR, PRED = OUT / "dataset", OUT / "model", OUT / "predictions"
for d in (DATA, MODEL_DIR, PRED):
    d.mkdir(parents=True, exist_ok=True)
print(MODE, "->", OUT)

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from angat_water import (acquire, dataset, infer, jrc, labels as lb, otsu, pipeline, pseudo_labels as pl,
                         sites, synthetic as syn, terramind_seg as tm)

CLS_CMAP = ListedColormap(["#c9b79c", "#2b7bba", "#f2c14e", "#222222"])         # land, water, uncertain, nodata
def cls_img(c):                      # map Otsu classes (0,1,2,255) to 0..3 for display
    return np.where(c == 255, 3, c)
LAB_CMAP = ListedColormap(["#c9b79c", "#2b7bba", "#dddddd"])                     # land, water, ignored
def lab_img(l):
    return np.where(l == pl.IGNORE, 2, l)
print("torch", tm.torch.__version__, "| cuda:", tm.torch.cuda.is_available())

## 2. Dataset: Sentinel-1 + JRC of the same month

For each Sentinel-1 date: Otsu classes (`map_water`), the JRC monthly map of the same calendar month on the same 10 m grid,
tiered labels (`labels.jrc_otsu_labels`), and the JRC-Otsu **agreement**. Scenes below `MIN_AGREEMENT` (wrong month, bad Otsu),
with too little JRC coverage, or without a water body are rejected and listed in `manifest.jsonl` with the reason.

In [ ]:
cfg = dataset.DatasetConfig(out_dir=str(DATA), start=START, end=END, max_month_offset=MAX_MONTH_OFFSET,
                            min_agreement=MIN_AGREEMENT, cache_dir=str(OUT / "cache"),
                            water=pipeline.WaterConfig(min_area_px=50 if MODE == "demo" else 200,
                                                       fill_holes_px=50 if MODE == "demo" else 200))
if (DATA / "manifest.jsonl").exists():
    (DATA / "manifest.jsonl").unlink()                  # start a fresh manifest (build_site appends)
GRIDS, SOURCES = {}, {}

if MODE == "real":
    jrc.init_gee(GEE_PROJECT)                           # `earthengine authenticate` once beforehand
    source = jrc.GeeJrc()
    for name in SITES:
        site = sites.get(name)
        cfg.water.bbox, cfg.water.crs = site.bbox, site.crs
        grid, scenes = dataset.iter_s1(site, cfg)
        GRIDS[name], SOURCES[name] = grid, source
        dataset.build_site(site, cfg, scenes, grid, source)
else:
    # Synthetic stand-ins: lakes with 4-look speckle, and a JRC that sees them at 30 m. Per site: four dates with a falling
    # water level (drawdown), one date whose JRC month shows a different level (must be rejected) and one without JRC.
    grid = jrc.Grid("EPSG:32651", 10.0, 300000.0, 1650000.0, 384, 384)
    for si, name in enumerate(SITES):
        truths, scenes = {}, []
        for k, (month, frac) in enumerate([("2019-03", .22), ("2019-04", .18), ("2019-05", .14), ("2019-06", .10)]):
            sc = syn.make_scene((384, 384), water_frac=frac, looks=4, seed=100 * si + k)
            truths[month] = sc["truth"]; scenes.append((f"{month}-12", sc["vv"], sc["vh"]))
        wrong = syn.make_scene((384, 384), water_frac=.20, looks=4, seed=100 * si + 8)
        truths["2019-07"] = syn.make_scene((384, 384), water_frac=.05, looks=4, seed=100 * si + 9)["truth"]
        scenes.append(("2019-07-12", wrong["vv"], wrong["vh"]))             # JRC July shows another level
        scenes.append(("2019-09-12", wrong["vv"], wrong["vh"]))             # no JRC data for September
        GRIDS[name], SOURCES[name] = grid, syn.SyntheticJrc(truths)
        dataset.build_site(sites.get(name), cfg, iter(scenes), grid, SOURCES[name])

In [ ]:
man = dataset.read_manifest(str(DATA))
tbl = pd.DataFrame([{
    "site": m["site"], "date": m["date"], "jrc_month": m.get("jrc_month"), "offset": m.get("month_offset"),
    "agreement": m.get("labels", {}).get("agreement"), "water_km2": (m["labels"]["water_px"] * RES_M**2 / 1e6) if "labels" in m else None,
    "jrc_nodata": m.get("labels", {}).get("jrc_nodata_frac"), "usable": m["usable"], "reject_reason": m.get("reject_reason")}
    for m in man])
display(tbl.round(3))
print(tbl.groupby("site")["usable"].agg(["sum", "count"]).rename(columns={"sum": "usable", "count": "scenes"}))
print("rejection reasons:", tbl["reject_reason"].value_counts().to_dict())

Look at one labelled scene. Grey in the label panel is **ignored** (JRC no-data, JRC-Otsu conflicts); the `source` panel shows
who decided each pixel: `agree` (both), `jrc_resolved` (JRC settled an Otsu-uncertain pixel), `otsu_edge` (10 m Otsu inside the
30 m JRC shoreline band).

In [ ]:
m0 = next(m for m in man if m["usable"])
z = np.load(DATA / m0["site"] / f"{m0['date']}.npz")
mm = jrc.matched_month(SOURCES[m0["site"]], GRIDS[m0["site"]], m0["date"], MAX_MONTH_OFFSET)
fig, ax = plt.subplots(1, 4, figsize=(18, 4.6))
ax[0].imshow(z["vv_db"], cmap="gray", vmin=-25, vmax=0);                     ax[0].set_title(f"VV dB (refined Lee)  {m0['site']} {m0['date']}")
ax[1].imshow(mm.water_class, cmap=ListedColormap(["#222", "#c9b79c", "#2b7bba"]), vmin=0, vmax=2, interpolation="nearest")
ax[1].set_title(f"JRC {mm.month:%Y-%m} (offset {mm.offset_months:+d})")
ax[2].imshow(lab_img(z["labels"]), cmap=LAB_CMAP, vmin=0, vmax=2, interpolation="nearest")
ax[2].set_title(f"labels (agreement {m0['labels']['agreement']:.3f})")
ax[3].imshow(z["source"], cmap="viridis", vmin=0, vmax=4, interpolation="nearest")
ax[3].set_title("source: 0 none 1 agree 2 jrc 3 otsu-edge")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()
print("label sources (px):", {k: v for k, v in m0["labels"].items() if k.endswith("_px")})

## 3. Patches and split

Patches with almost no water are subsampled (`KEEP_EMPTY`) so the rare class is not drowned. With `HOLDOUT` sites the validation set is
whole sites the model never saw - the honest test of generalisation (train Angat, test Magat).

In [ ]:
X, Y, metas = dataset.load_patches(str(DATA), size=PATCH, stride=PATCH // 2, keep_empty=KEEP_EMPTY,
                                   max_month_offset=MAX_MONTH_OFFSET)
tr, va = dataset.split_patches(metas, PATCH, HOLDOUT)
lab = Y[Y != pl.IGNORE]
print(f"{len(X)} patches {X.shape[1:]} | train {len(tr)} | val {len(va)} | water share of labelled pixels {np.mean(lab == 1):.3f}")
print("per site:", pd.Series([m['site'] for m in metas]).value_counts().to_dict())
assert len(tr) and len(va), "need both train and validation patches: check HOLDOUT / the manifest"

## 4. Train TerraMind

`TerraMindWater` = TerraMind encoder (`S1RTC`: terrain-corrected gamma0 in dB, standardised with TerraMind's own statistics) +
multi-layer token decoder + a small pixel stem for sharp shorelines. The encoder is frozen by default. Loss = cross-entropy (ignored
pixels excluded) + Dice. The best epoch by validation IoU is kept.

In [ ]:
tm.torch.manual_seed(0); np.random.seed(0)
model = tm.TerraMindWater(BACKBONE, pretrained=PRETRAINED, unfreeze_last=UNFREEZE_LAST)
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{BACKBONE}: {n_train/1e6:.2f}M trainable of {sum(p.numel() for p in model.parameters())/1e6:.1f}M parameters")
state, hist = tm.train(model, X, Y, tr, va, epochs=EPOCHS, batch=BATCH, lr=LR)

h = pd.DataFrame(hist)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(h.epoch + 1, h.loss, marker="o"); ax[0].set_title("training loss"); ax[0].set_xlabel("epoch")
ax[1].plot(h.epoch + 1, h.val_iou, marker="o", color="C1"); ax[1].set_title("validation IoU (water)"); ax[1].set_xlabel("epoch")
plt.tight_layout(); plt.show()

CKPT = MODEL_DIR / "terramind_water.pt"
tm.save_checkpoint(model, str(CKPT), {"sites": SITES, "holdout": HOLDOUT, "labels": "jrc_monthly x otsu", "mode": MODE})
print("saved", CKPT)

### Validation per site

IoU is measured against the **training labels** (JRC x Otsu on the labelled pixels only), so it shows how well the model reproduces them on a
site it has not seen. It is not accuracy against reality: for that, compare with an independent reference (a clear Sentinel-2 date, OPERA DSWx,
or hand-checked patches).

In [ ]:
Xt, Yt = tm.torch.from_numpy(X), tm.torch.from_numpy(Y.astype(np.int64))
for s in sorted({metas[i]["site"] for i in va}):
    idx = [i for i in va if metas[i]["site"] == s]
    print(f"{s:>8}: IoU(water) vs labels = {tm.evaluate(model, Xt, Yt, idx):.4f}  ({len(idx)} patches)")

## 5. Inference on a scene

Reload the checkpoint (no pretrained download needed - the weights are inside), run **Otsu**, the **TerraMind probability**, and the **fusion**:
Otsu keeps the 10 m boundary where it is certain, the model decides the pixels Otsu is unsure about, and strong disagreements are flagged
rather than silently overridden (`model-only` = candidate flooded vegetation / shallow water; `Otsu-only` = water the model doubts).

Real mode reads the first Sentinel-1 date in `INFER_FROM..INFER_TO` of `INFER_SITE`; these dates may be after 2021.

In [ ]:
model2, meta = tm.load_checkpoint(str(CKPT))
print("checkpoint meta:", meta)

if MODE == "real":
    site = sites.get(INFER_SITE)
    cfg.water.bbox, cfg.water.crs = site.bbox, site.crs
    GEOBOX, scenes = infer.iter_window(site, INFER_FROM, INFER_TO, cfg.water, cache_dir=str(OUT / "cache"))
    day, vv, vh = next(scenes); TRUTH = None
else:
    from affine import Affine
    class GEOBOX:                                              # georeferencing for the synthetic export
        transform = Affine(10, 0, 300000, 0, -10, 1650000); crs = "EPSG:32651"
    sc = syn.make_scene((384, 384), water_frac=0.12, looks=4, seed=999)
    day, vv, vh, TRUTH = "2024-02-14", sc["vv"], sc["vh"], sc["truth"]

res = infer.infer_scene(vv, vh, model2, cfg.water, tile=PATCH, overlap=PATCH // 4)
print(day, "| Otsu mode:", res.thresholds.mode, "| t_edge", round(res.thresholds.t_edge, 2), "| t_binary", round(res.thresholds.t_binary, 2))
print({k: round(v, 3) for k, v in res.area_km2(RES_M).items()}, "km2")
if TRUTH is not None:
    print("IoU vs synthetic truth -> Otsu: %.3f | TerraMind: %.3f | fused: %.3f" % (
        syn.iou(res.otsu_water, TRUTH), syn.iou(res.prob > .5, TRUTH), syn.iou(res.fused_water, TRUTH)))

In [ ]:
FLAG_CMAP = ListedColormap(["#c9b79c", "#2b7bba", "#e4572e", "#7b2cbf"])
fig, ax = plt.subplots(1, 5, figsize=(22, 4.6))
ax[0].imshow(res.vv_db, cmap="gray", vmin=-25, vmax=0);                         ax[0].set_title(f"VV dB  {day}")
ax[1].imshow(cls_img(res.classes), cmap=CLS_CMAP, vmin=0, vmax=3, interpolation="nearest"); ax[1].set_title("Otsu classes (yellow = uncertain)")
im = ax[2].imshow(res.prob, cmap="Blues", vmin=0, vmax=1);                       ax[2].set_title("TerraMind P(water)"); plt.colorbar(im, ax=ax[2], fraction=.046)
ax[3].imshow(res.fused_water, cmap="Blues", interpolation="nearest");           ax[3].set_title("fused water")
ax[4].imshow(res.flags, cmap=FLAG_CMAP, vmin=0, vmax=3, interpolation="nearest"); ax[4].set_title("flags: red model-only, purple Otsu-only")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## 6. Water-area time series

Surface area per date (km2) from Otsu alone and from the fused map. A reservoir's level should change smoothly; jumps usually mean a
bad pass (wind, rain, a different orbit) and are worth inspecting. In `real` mode this reads every pass in `INFER_FROM..INFER_TO`.

In [ ]:
rows = []
if RUN_TIMESERIES:
    if MODE == "real":
        _, stream = infer.iter_window(sites.get(INFER_SITE), INFER_FROM, INFER_TO, cfg.water, cache_dir=str(OUT / "cache"))
    else:                                                       # drawdown: the water level falls from 20 % to 7 % of the scene
        demo = [syn.make_scene((384, 384), water_frac=f, looks=4, seed=500 + k) for k, f in enumerate([.20, .16, .12, .09, .07])]
        stream = ((f"2024-0{k+1}-10", s["vv"], s["vh"]) for k, s in enumerate(demo))
    for d, v, h_ in stream:
        r = infer.infer_scene(v, h_, model2, cfg.water, tile=PATCH, overlap=PATCH // 4)
        rows.append({"date": pd.Timestamp(str(d)), **r.area_km2(RES_M), "otsu_mode": r.thresholds.mode})
ts = pd.DataFrame(rows)
if len(ts):
    display(ts.set_index("date").round(3))
    ax = ts.plot(x="date", y=["otsu", "fused"], marker="o", figsize=(8, 3.2), ylabel="water area (km2)", title=f"{INFER_SITE}: surface water")
    plt.tight_layout(); plt.show()

## 7. Export

GeoTIFFs for the scene above: `*_otsu_classes` (0 land, 1 water, 2 uncertain, 255 no data), `*_fused_water` (0/1), `*_flags`, `*_prob`.

In [ ]:
paths = infer.write_result(str(PRED), str(day), res, GEOBOX)
for k, p in paths.items():
    print(f"{k:>14}: {p}")

## Notes

- **Labels are not ground truth.** They are where JRC (Landsat, monthly, 30 m) and Otsu (Sentinel-1, same date, 10 m) agree. Check the model against an independent
  reference before relying on it, and look at the `model-only` / `Otsu-only` flags: they are where the two disagree.
- **Watch the agreement column.** Low agreement on a scene usually means the JRC month does not match the Sentinel-1 date (the reservoir moved within the month).
- **Radar shadow** on steep terrain around reservoirs gives false water; a HAND/slope mask is not included yet.
- Rivers a few pixels wide are mostly mixed pixels at 10 m and are the weakest case; add river sites (`sites.custom`) and keep `KEEP_EMPTY` low so they are not swamped by land.
- `MODE = "demo"` uses a random-init tiny model on synthetic scenes: it checks that the code runs, nothing more.